# Chapter 4: Biological Patterns and Regular Expressions

**Colab slideshow notebook**  
Based on Chapter 4 of the book "Computational Thinking for Life Scientists"
by Benny Chor and Amir Rubinstein.

## Chapter goals

By the end of this chapter, we want to understand:

- the difference between **exact search** and **pattern (motif) search**,
- how to describe a biological motif with a **regular expression**,
- how to use Python's `re` module: `search`, `finditer`, `findall`, `sub`, `compile`,
- why repetition operators are **greedy**, and how to make them non-greedy,
- what a **finite state machine (FSM)** is, and how it relates to regular expressions.

## 4.1 Pattern matching — the exact string searching problem

**Input:** a string `p` of length `m` (the *pattern*) and a string `s` of length `n` (the *text*), where `m ≤ n`.

**Output:** does `s` contain `p` as a substring?

Two simple solutions:

- **Naïve:** compare `p` to every substring of `s` of length `m` → `O(n·m)` time.
- **Hash table:** put every length-`m` substring of `s` in a `set` → also `O(n·m)` to build,
  and more memory. Worth it only when we search **many** patterns of the same length in the same `s`
  (the text is *pre-processed* once).

In [ ]:
# Code snippet: naïve exact search

def exact_search(p, s):
    ''' return the first index where pattern p appears in text s, or -1 '''
    m = len(p)
    for i in range(len(s) - m + 1):
        if s[i:i+m] == p:
            return i
    return -1

dna = "CCCTATAAAAATTCCC"
print('exact_search("TATAA", dna) =', exact_search("TATAA", dna))
print('exact_search("GGG", dna)   =', exact_search("GGG", dna))
print('dna.find("TATAA")          =', dna.find("TATAA"))   # Python's built-in method

## A twist: motifs with "degrees of freedom"

Many biological motifs match **many** possible strings:

- **TATA-box:** `TATAA`, then any three nucleotides, then `TT`
- **N-glycosylation site:** Asn (`N`), then any amino acid but Pro, then Ser or Thr, then any amino acid but Pro
- **ORF (open reading frame):** `ATG`, then `3n` nucleotides (n ≥ 0), then a stop codon `TAG` / `TAA` / `TGA`

This is the **pattern matching** (or **motif search**) problem.

In [ ]:
# Code snippet: searching a TATA-box "by hand" — works, but cumbersome

def find_tata_by_hand(s):
    ''' return the first index of a TATA-box (TATAA + 3 nucleotides + TT), or -1 '''
    for i in range(len(s) - 9):
        if s[i:i+5] == "TATAA" and s[i+5] in "ACGT" and s[i+6] in "ACGT" \
                and s[i+7] in "ACGT" and s[i+8:i+10] == "TT":
            return i
    return -1

print('find_tata_by_hand("CCCTATAAAAATTCCC") =', find_tata_by_hand("CCCTATAAAAATTCCC"))
print('find_tata_by_hand("CCCCCCCCCCCCCCCC") =', find_tata_by_hand("CCCCCCCCCCCCCCCC"))

## 4.2 Regular expressions

A **regular expression** (RE) is a compact way to describe a textual pattern.

- Expressive enough for many biological motifs
- A central notion in the theory of computing (closely related to *finite state machines*, section 4.3)
- Python has a built-in module for regular expressions: `re`

In [ ]:
# Code snippet: a first search with the re module
import re

match = re.search("hello", "Hello dogs, hello cats!")
print("match.group() =", match.group())   # the matching string
print("match.start() =", match.start())   # where the match starts
print("match.end()   =", match.end())     # one position after the match ends
print("match.span()  =", match.span())    # (start, end)

In [ ]:
# Code snippet: no match -> re.search returns None

match = re.search("Goodbye", "Hello dogs, hello cats!")
print("match =", match)

## Several options at one position: `[ ]`

`"[Hh]ello"` means: `H` **or** `h`, followed by `ello`.

- `re.search` returns only the **first** match
- `re.finditer` returns **all** matches, as match objects (with `group`, `span`, …)
- `re.findall` returns a **list of the matching strings** only (no positions)

In [ ]:
# Code snippet: first match vs. all matches

text = "Hello dogs, hello cats!"

match = re.search("[Hh]ello", text)
print("re.search  ->", match.group(), match.span())   # first match only

print("re.finditer ->")
for m in re.finditer("[Hh]ello", text):
    print("   ", m.group(), m.span())

print("re.findall ->", re.findall("[Hh]ello", text))

## Exercise 1

Write a function `find_motifs(pattern, seq)` that prints every match, with its start and end
positions, and then the number of matches found.

```python
>>> find_motifs("[Hh]ello", "Hello dogs, hello cats!")
Hello from 0 to 5
hello from 12 to 17
2 matches found
```

In [ ]:
# Exercise 1: your solution here

def find_motifs(pattern, seq):
    pass  # replace with your code

find_motifs("[Hh]ello", "Hello dogs, hello cats!")

## 4.2.2 Meta-characters

| Meta-character | Meaning | Example |
|---|---|---|
| `[ ]` | a set of possible characters | `"[ABC]"` matches `A`, `B` or `C`; `"[A-Z]"` any uppercase letter |
| `[^ ]` | complement of a set | `"[^R]"` matches any character except `R` |
| `.` | any single character (except new-line) | `"ATT.T"` matches `ATTCT`, `ATTGT` |
| `\|` | "or" | `"ATT\|CG\|C"` matches `ATT`, `CG` or `C` |
| `( )` | grouping | `"AT(T\|CG\|C)"` matches `ATT`, `ATCG` or `ATC` |
| `*` | 0 or more occurrences | `"AT*"` matches `A`, `AT`, `ATT`, … |
| `+` | 1 or more occurrences | `"AT+"` matches `AT`, `ATT`, … but not `A` |
| `?` | 0 or 1 occurrences | `"AT?"` matches `A` or `AT` |
| `{n}` | exactly n occurrences | `"(ATTG){3}"` matches `ATTGATTGATTG` |
| `{m,n}` | between m and n occurrences | `"(AT){3,5}"` matches `ATATAT` … `ATATATATAT` |

In [ ]:
# Code snippet: try the meta-characters (fullmatch = the WHOLE string must match the pattern)

tests = [
    ("ATT.T",     ["ATTCT", "ATTGT", "ATTTCT"]),
    ("ATT|CG|C",  ["ATT", "CG", "C", "AT"]),
    ("AT(T|CG|C)",["ATT", "ATCG", "ATC", "ATG"]),
    ("AT*",       ["A", "AT", "ATT", "TT"]),
    ("AT+",       ["A", "AT", "ATTT"]),
    ("AT?",       ["A", "AT", "ATT"]),
    ("(ATTG){3}", ["ATTGATTGATTG", "ATTGATTG"]),
    ("(AT){3,5}", ["ATATAT", "ATATATATAT", "ATATTATAT"]),
]

for pattern, strings in tests:
    for s in strings:
        print(f"{pattern:12} {s:14} ->", re.fullmatch(pattern, s) is not None)
    print()

In [ ]:
# Code snippet: the N-glycosylation motif as a regular expression
# N, then any amino acid but P, then S or T, then any amino acid but P

protein = "MKNGSAVNPTLNVTQ"
for m in re.finditer("N[^P][ST][^P]", protein):
    print(m.group(), "at", m.span())

## Exercise 2

Write a function `has_tata(dna)` that checks whether a DNA string contains a TATA-box-like
pattern: `TATAA`, then any three nucleotides, then `TT`.

```python
>>> has_tata("CCCTATAAAAATTCCC")
True
>>> has_tata("CCCCCCCCCCCCCCCC")
False
```

In [ ]:
# Exercise 2: your solution here

def has_tata(dna):
    pass  # replace with your code

print('has_tata("CCCTATAAAAATTCCC") =', has_tata("CCCTATAAAAATTCCC"))
print('has_tata("CCCCCCCCCCCCCCCC") =', has_tata("CCCCCCCCCCCCCCCC"))

## Anchors: beginning and end of the sequence

| Meta-character | Meaning | Example |
|---|---|---|
| `^` | beginning of the sequence | `"^AUG"` matches `AUGAGC` but not `AAUGC` |
| `$` | end of the sequence | `"UAA$"` matches `AGCUAA` but not `ACUAAG` |

Careful: inside `[ ]`, `^` means "not" (complement).

**Task:** is a given sequence an ORF — does it *start* with a start codon and *end* with a stop codon?

In [ ]:
# Code snippet: three attempts to recognise an ORF

def bad_is_orf(seq):
    # checks only that seq CONTAINS an ORF-like piece
    return re.search("ATG[ACGT]*(TAG|TAA|TGA)", seq) != None

def is_orf_almost(seq):
    # must start with ATG and end with a stop codon — but length is not checked
    return re.search("^ATG[ACGT]*(TAG|TAA|TGA)$", seq) != None

def is_orf(seq):
    # the middle must be made of whole codons: ([ACGT]{3})*
    return re.search("^ATG([ACGT]{3})*(TAG|TAA|TGA)$", seq) != None

for seq in ["CCATGCCCTAACC", "ATGCCTAA", "ATGCCCTAA"]:
    print(f"{seq:15} bad_is_orf={bad_is_orf(seq)!s:5}  is_orf_almost={is_orf_almost(seq)!s:5}  is_orf={is_orf(seq)}")

In [ ]:
# Code snippet: is_orf is still not perfect — a stop codon in the middle

seq = "ATGTAATAA"
print("is_orf(\"ATGTAATAA\") =", is_orf(seq))   # True, although TAA appears in the middle

In [ ]:
# Code snippet: replace every occurrence of a pattern with re.sub

def replace_tata(seq):
    return re.sub("TATAA[ACGT]{3}TT", "-tata-", seq)

seq = "GGTTCGGATGATGTATAAGGCTTGGTATAAAAATTGGCTTATA"
print("before:", seq)
print("after: ", replace_tata(seq))

## 4.2.3 Length matters! Greedy matching

**Goal:** find the positions of *all* ORFs within a sequence.

Repetition operators such as `*` and `+` are **greedy**: the matching engine tries the **longest**
possible repeat first.

In [ ]:
# Code snippet: a greedy search finds one long "ORF" instead of two

def find_all_orfs_greedy(seq):
    for orf in re.finditer("ATG([ACGT]{3})*(TAG|TAA|TGA)", seq):
        print(orf.start(), orf.end())

print("one ORF:")
find_all_orfs_greedy("CCCATGCCCTAACCC")
print("two ORFs:")
find_all_orfs_greedy("CCCATGCCCCCCTAACCCCCCATGCCCCCCTAACCC")   # only one (long) match!

## Non-greedy repetition: `*?` and `+?`

Adding `?` after a repetition operator asks for the **shortest** possible match.

In [ ]:
# Code snippet: non-greedy search finds both ORFs

def find_all_orfs(seq):
    for orf in re.finditer("ATG([ACGT]{3})*?(TAG|TAA|TGA)", seq):
        print(orf.start(), orf.end())

find_all_orfs("CCCATGCCCCCCTAACCCCCCATGCCCCCCTAACCC")

print('greedy:     ', re.findall("a[ac]+c", "aaaaccccaaaaaccccc"))
print('non-greedy: ', re.findall("a[ac]+?c", "aaaaccccaaaaaccccc"))

## 4.2.4 Classes of characters

| String | Class | Equivalent |
|---|---|---|
| `\d` | decimal digit | `[0-9]` |
| `\D` | non-digit | `[^0-9]` |
| `\s` | any whitespace | `[ \t\n\r\f\v]` |
| `\S` | non-whitespace | `[^ \t\n\r\f\v]` |
| `\w` | any alphanumeric | `[a-zA-Z0-9_]` |
| `\W` | non-alphanumeric | `[^a-zA-Z0-9_]` |

**Tip:** write patterns that contain `\` as *raw strings*: `r"\d+"`.
In a normal string, Python itself would try to interpret the backslash first.

In [ ]:
# Code snippet: does a string represent a floating-point number such as 3.14?
# digits, a dot, digits — and nothing else (^ ... $)

float_pattern = r"^\d+\.\d+$"

for s in ["3.14", "100.0", "3.", "3,14", "abc"]:
    print(f"{s:6} ->", re.search(float_pattern, s) != None)

## 4.2.5 Compiling a pattern for reuse

**Compiling** a pattern = pre-processing it once, so that many later searches are faster.

Useful when the same pattern is searched in many different sequences.
Compiled patterns have the same methods: `search`, `findall`, `finditer`, …

In [ ]:
# Code snippet: compile once, use many times

pattern = re.compile("[Hh]ello")   # pattern is a "compiled" regular expression
print(pattern.findall("Hello dogs, hello cats!"))
print(pattern.findall("Hello world"))

## 4.3 Finite state machines (FSM)

A **finite state machine** is a simple mathematical model:

- a finite **alphabet** Σ, e.g. `{A, C, G, T}`
- a finite set of **states**, e.g. `{q0, q1}`
- an **initial state** (`q0`)
- a set of **accepting** ("final") states, e.g. `{q1}`
- **transition rules**: from a state, reading a character, move to a state

The machine reads the input one character at a time. It **accepts** the input if it ends in an accepting state.

## Example: an FSM with two states

| from \ move to | q0 | q1 |
|---|---|---|
| **q0** | C, T, G | A |
| **q1** | C, T, G | A |

- Start at `q0`; reading `A` moves to `q1`; reading `C`, `T` or `G` moves to `q0`.
- `q1` is the accepting state.

Which strings does it accept? Let's simulate it in Python.

In [ ]:
# Code snippet: simulating a finite state machine

# transitions[state][character] = next state
transitions = {
    "q0": {"A": "q1", "C": "q0", "G": "q0", "T": "q0"},
    "q1": {"A": "q1", "C": "q0", "G": "q0", "T": "q0"},
}
start = "q0"
accepting = {"q1"}

def fsm_accepts(word, transitions, start, accepting):
    ''' walk the word through the machine; accept iff we end in an accepting state '''
    state = start
    for ch in word:
        state = transitions[state][ch]
    return state in accepting

for word in ["AAAC", "AAA", "CA", "", "GATTACA"]:
    print(f"{word!r:10} accepted: {fsm_accepts(word, transitions, start, accepting)}")

## FSMs and regular expressions

This machine accepts **exactly the strings that end with `A`** — the same pattern as the regular expression `A$`.

A fundamental result of computer science:
**every pattern that can be described by a regular expression can be described by an FSM, and vice versa.**

FSMs are also called *deterministic finite automata* (DFA): given the current state and the next character,
there is always exactly one state to move to.

In [ ]:
# Code snippet: the FSM and the regular expression "A$" agree

import itertools

words = ["".join(w) for k in range(1, 5) for w in itertools.product("ACGT", repeat=k)]
agree = all(fsm_accepts(w, transitions, start, accepting) == (re.search("A$", w) != None)
            for w in words)
print("number of words checked:", len(words))
print("FSM and regex agree on all of them:", agree)

## Exercise 3

(a) Draw an FSM that accepts all DNA strings that **contain** a codon for Arginine
(`CGT`, `CGA`, `CGC`, `CGG`, `AGA`, `AGG`). Use as few states as you can (6 should suffice).
Mark the initial and accepting state(s).

(b) Write a regular expression in Python's syntax for the same pattern.

(c) Implement your machine as a `transitions` dictionary and check it against your regular
expression with `fsm_accepts`, as in the previous slide.

## Limits of regular expressions

FSMs also teach us what regular expressions **cannot** do.

There is no regular expression for the pattern `aⁿbⁿ`: some number of `a`s followed by
**exactly the same number** of `b`s. (Intuition: a machine with finitely many states cannot "count"
arbitrarily high. The proof is omitted.)

FSMs are also useful models of simple devices — e.g. a vending machine whose state remembers how much money was inserted.

## Reflection

- Regular expressions describe patterns with "degrees of freedom", repetitions, alternatives.
- Python's `re` syntax is very similar to the syntax of other languages and tools.
- A regular expression carries **no probabilities**: every TATA-box variant is "equally good",
  although in nature some variants are much more common.
  Biologists use **sequence logos** to show how common each character is at each position.

## Challenge Yourself — Problem 1: Digestion by restriction enzymes

Two restriction enzymes, X and Y:

- X recognises `ANT-AATTTTT`
- Y recognises `GCRW-TGGGGG`

(`N` = any base, `R` = A or G, `W` = A or T, and `-` marks the cut site.)

(a) Predict the fragment lengths when the genome of *Mycobacterium leprae* is digested by X.
Example: `CCCAATAATTTTTGGGATTAATTTTTGG` is cut into `CCCAAT`, `AATTTTTGGGATT`, `AATTTTTGG` → lengths 6, 13, 9.

(b) Repeat when X and Y digest the DNA together.

In [ ]:
# Code snippet: load the Mycobacterium leprae genome from the course repository
import os
import urllib.request

os.makedirs("data", exist_ok=True)
BASE_URL = "https://raw.githubusercontent.com/ariefouren/BIU-bio-python-2026/main/data"
filename = "Mycobacterium_leprae.txt"
local_path = f"data/{filename}"

if not os.path.exists(local_path):
    urllib.request.urlretrieve(f"{BASE_URL}/{filename}", local_path)
    print("Downloaded:", filename)

with open(local_path) as f:
    Mycobacterium_leprae = f.read().replace("\n", "")

print("genome length:", len(Mycobacterium_leprae))

In [ ]:
# Problem 1: your solution here
# Hint: m.start() of the recognition site + the offset of "-" gives the cut position.

def fragment_lengths(seq, pattern, cut_offset):
    pass  # replace with your code

print(fragment_lengths("CCCAATAATTTTTGGGATTAATTTTTGG", "A[ACGT]TAATTTTT", 3))   # expected: [6, 13, 9]

## Challenge Yourself — Problem 2: printing the "context" of a pattern

(a) Write `find_pattern(seq, pattern, pre, post)` that prints every occurrence of `pattern` in `seq`
in its local context: `pre` characters before and `post` characters after (fewer at the ends of `seq`).
Print the pattern in UPPERCASE, the context in lowercase, and the start position.

```python
>>> find_pattern(CAP2, "NM", 5, 5)
maNMqglve starts at position 2
rnrgsNMfnhls starts at position 127
```

(b) CAP2 contains this pattern: two positions from L/I/V/M, any amino acid, R, L, D or E,
any four amino acids, then R, L, E. Find it with your function.

In [ ]:
# Code snippet: the beginning of protein CAP2 of the Orangutan (the excerpt shown in the book)

CAP2 = ("MANMQGLVERLERAVSRLESLSAESHRPPGNCGEVNGVIGGVAPSVEAFDKLMDSMVAEF"
        "LKNSRILAGDVETHAEMVHSAFQAQRAFLLMASQYQQPHENDVAALLKPISEKIQEIQTF"
        "RERNRGSNMFNHLSAVSESIPALGWIAVSPKPGPYVKEMNDAATFYTNRVLKDYKHSDLR"
        "HVDWVKSYLNIWSELQAYIKEHHTTGLTWSKTGPVASTVSAFSVLSSGPGLPPPPPPPPP")

def find_pattern(seq, pattern, pre, post):
    pass  # replace with your code

find_pattern(CAP2, "NM", 5, 5)

## Chapter 4 Summary

In this chapter we studied:

- exact search vs. pattern (motif) search
- regular expressions and Python's `re` module: `search`, `finditer`, `findall`, `sub`, `compile`
- meta-characters, anchors `^` `$`, and character classes `\d` `\w` `\s`
- greedy (`*`, `+`) vs. non-greedy (`*?`, `+?`) repetition — finding all ORFs
- finite state machines, and their equivalence to regular expressions
- a limit of regular expressions: no RE for `aⁿbⁿ`